In [36]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier

import joblib
import time
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')


In [37]:
print("Loading dataset...")
df = pd.read_csv('balanced_urls.csv')  
print(df.head())
print(df.columns)

Loading dataset...
                         url   label  result
0     https://www.google.com  benign       0
1    https://www.youtube.com  benign       0
2   https://www.facebook.com  benign       0
3      https://www.baidu.com  benign       0
4  https://www.wikipedia.org  benign       0
Index(['url', 'label', 'result'], dtype='object')


In [38]:
print("\n[2/8] Checking 'result' column distribution...")

# Check if 'result' column exists
if 'result' not in df.columns:
    print("ERROR: 'result' column not found!")
    print(f"Available columns: {df.columns.tolist()}")
    print("\nPlease use the correct column name or rename your column to 'result'")
    exit()


[2/8] Checking 'result' column distribution...


In [39]:
result_counts = df['result'].value_counts().sort_index()
total = len(df)

# Get counts
count_0 = result_counts.get(0, 0)
count_1 = result_counts.get(1, 0)
percentage_0 = (count_0 / total) * 100
percentage_1 = (count_1 / total) * 100

In [40]:
print("\n📊 RESULT COLUMN DISTRIBUTION:")
print("="*40)
print(f"Class 0 (Benign/Misc):     {count_0:8,} URLs  ({percentage_0:5.1f}%)")
print(f"Class 1 (Malicious/Spam):  {count_1:8,} URLs  ({percentage_1:5.1f}%)")
print(f"Total:                     {total:8,} URLs  (100.0%)")
print("="*40)


📊 RESULT COLUMN DISTRIBUTION:
Class 0 (Benign/Misc):      316,254 URLs  ( 50.0%)
Class 1 (Malicious/Spam):   316,254 URLs  ( 50.0%)
Total:                      632,508 URLs  (100.0%)


In [62]:
from urllib.parse import urlparse
import re

def normalize_url(url):
    """Normalize URL to reduce variations"""
    # Convert to lowercase
    url = url.lower()
    
    # Add protocol if missing
    if not url.startswith(('http://', 'https://')):
        url = 'http://' + url
    
    # Parse URL
    parsed = urlparse(url)
    
    # Remove 'www' subdomain
    hostname = parsed.hostname or ''
    if hostname.startswith('www.'):
        hostname = hostname[4:]
    
    # Remove default ports
    netloc = hostname
    if parsed.port:
        if (parsed.scheme == 'http' and parsed.port == 80) or \
           (parsed.scheme == 'https' and parsed.port == 443):
            netloc = hostname
        else:
            netloc = f"{hostname}:{parsed.port}"
    
    # Rebuild URL
    normalized = f"{parsed.scheme}://{netloc}{parsed.path}"
    
    # Remove trailing slash
    normalized = re.sub(r'/$', '', normalized)
    
    return normalized

# Update your feature extraction
def extract_features(url):
    url = normalize_url(url)
    # ... rest of your feature extraction

In [63]:
from urllib.parse import urlparse

def get_domain(url):
    """Extract only the domain name"""
    parsed = urlparse(url)
    domain = parsed.hostname or ''
    # Remove www
    if domain.startswith('www.'):
        domain = domain[4:]
    return domain

# Use domain for classification instead of full URL
domain = get_domain(url)
score, verdict, risk = classify_url(domain)

[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [64]:
def preprocess_url(url):
    """Comprehensive URL preprocessing"""
    # Lowercase
    url = url.lower()
    
    # Add protocol
    if not url.startswith(('http://', 'https://')):
        url = 'https://' + url
    
    # Parse
    parsed = urlparse(url)
    
    # Normalize hostname
    hostname = parsed.hostname or ''
    hostname = re.sub(r'^www\.', '', hostname)  # Remove www
    
    # Normalize path (remove default index files)
    path = parsed.path
    path = re.sub(r'/index\.(html?|php|asp)$', '/', path)
    
    # Remove trailing slash
    if path.endswith('/') and len(path) > 1:
        path = path[:-1]
    
    # Reconstruct
    normalized = f"{parsed.scheme}://{hostname}{path}"
    
    # Add query parameters in sorted order
    if parsed.query:
        params = sorted(parsed.query.split('&'))
        normalized += '?' + '&'.join(params)
    
    return normalized

# Apply to all URLs
df['normalized_url'] = df['url'].apply(preprocess_url)

In [65]:
# ==================== 4. TRAIN RANDOM FOREST ====================
# Use n_jobs=-1 for parallel processing; n_estimators=100 as a good balance
print("Training Random Forest...")
start_time = time.time()

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,           # prevents overfitting
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    n_jobs=-1,              # use all CPU cores
    random_state=42,
    verbose=1
)

rf_model.fit(X_train_tfidf, y_train)
print(f"Training time: {time.time() - start_time:.2f} seconds")

Training Random Forest...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:   35.7s


Training time: 109.93 seconds


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  1.7min finished


In [66]:
unique_values = df['result'].unique()
print(f"\n✓ Unique values in 'result' column: {sorted(unique_values)}")

# Validate values are only 0 and 1
invalid_values = df[~df['result'].isin([0, 1])]
if len(invalid_values) > 0:
    print(f"\n⚠️ WARNING: Found {len(invalid_values)} rows with values other than 0 or 1!")
    print(f"Invalid values: {invalid_values['result'].unique()}")
    print("Fixing: Removing invalid rows...")
    df = df[df['result'].isin([0, 1])]
    print(f"✓ Removed invalid rows. New total: {len(df)}")


✓ Unique values in 'result' column: [np.int64(0), np.int64(1)]


In [67]:
balance_ratio = min(count_0, count_1) / max(count_0, count_1)

print(f"\n📈 Balance Ratio: {balance_ratio:.3f}")
if balance_ratio >= 0.8:
    print("✅ EXCELLENT: Dataset is well-balanced (80-100% balance)")
elif balance_ratio >= 0.6:
    print("👍 GOOD: Dataset is moderately balanced (60-80% balance)")
elif balance_ratio >= 0.4:
    print("⚠️ FAIR: Dataset is somewhat imbalanced (40-60% balance)")
    print("   Consider using class weights for better accuracy")
else:
    print("❌ POOR: Dataset is highly imbalanced (<40% balance)")
    print("   Recommend using SMOTE or class weights")


📈 Balance Ratio: 1.000
✅ EXCELLENT: Dataset is well-balanced (80-100% balance)


In [68]:
print("\n[3/8] Identifying URL column...")

url_column = None
for col in ['url', 'URL', 'link', 'Link', 'website', 'Website', 'address']:
    if col in df.columns:
        url_column = col
        break

if url_column is None:
    print("❌ ERROR: No URL column found!")
    print(f"Available columns: {df.columns.tolist()}")
    exit()

print(f"✓ Using '{url_column}' as URL column")


[3/8] Identifying URL column...
✓ Using 'url' as URL column


In [69]:
missing_urls = df[url_column].isna().sum()
if missing_urls > 0:
    print(f"⚠️ Found {missing_urls} missing URLs. Removing...")
    df = df.dropna(subset=[url_column])

# ============================================
# STEP 4: DATA QUALITY REPORT
# ============================================
print("\n[4/8] Data Quality Report...")
print(f"✓ Final dataset size: {len(df)} URLs")
print(f"✓ Class 0 (Benign): {(df['result']==0).sum()}")
print(f"✓ Class 1 (Malicious): {(df['result']==1).sum()}")


[4/8] Data Quality Report...
✓ Final dataset size: 632508 URLs
✓ Class 0 (Benign): 316254
✓ Class 1 (Malicious): 316254


In [70]:
sample_benign = df[df['result']==0][url_column].head(3).tolist()
sample_malicious = df[df['result']==1][url_column].head(3).tolist()

print("\n📝 Sample URLs from your dataset:")
print("\nBenign URLs (Class 0):")
for url in sample_benign:
    print(f"  • {url[:80]}...")
print("\nMalicious URLs (Class 1):")
for url in sample_malicious:
    print(f"  • {url[:80]}...")


📝 Sample URLs from your dataset:

Benign URLs (Class 0):
  • https://www.google.com...
  • https://www.youtube.com...
  • https://www.facebook.com...

Malicious URLs (Class 1):
  • http://atualizacaodedados.online...
  • http://webmasteradmin.ukit.me/...
  • http://stcdxmt.bigperl.in/klxtv/apps/uk/...


In [71]:
print("\n[5/8] Preparing features and labels...")

X = df[url_column].astype(str)
y = df['result'].values

# Verify final distribution
print(f"\n✓ Final class distribution:")
print(f"  Class 0: {(y==0).sum():,} URLs")
print(f"  Class 1: {(y==1).sum():,} URLs")
print(f"  Total:   {len(y):,} URLs")

# ============================================
# STEP 6: TRAIN/TEST SPLIT
# ============================================
print("\n[6/8] Splitting data (80% train, 20% test)...")


[5/8] Preparing features and labels...

✓ Final class distribution:
  Class 0: 316,254 URLs
  Class 1: 316,254 URLs
  Total:   632,508 URLs

[6/8] Splitting data (80% train, 20% test)...


In [72]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  # Maintains class balance
)

print(f"✓ Training set: {len(X_train):,} URLs")
print(f"  - Class 0: {(y_train==0).sum():,}")
print(f"  - Class 1: {(y_train==1).sum():,}")
print(f"✓ Testing set: {len(X_test):,} URLs")
print(f"  - Class 0: {(y_test==0).sum():,}")
print(f"  - Class 1: {(y_test==1).sum():,}")

✓ Training set: 506,006 URLs
  - Class 0: 253,003
  - Class 1: 253,003
✓ Testing set: 126,502 URLs
  - Class 0: 63,251
  - Class 1: 63,251


In [73]:
print("\n[7/8] Converting URLs to features (TF-IDF)...")
print("  This may take 2-5 minutes for large datasets...")

vectorizer = TfidfVectorizer(
    analyzer='char',        # Character-level analysis
    ngram_range=(3, 6),     # 3-6 character sequences
    max_features=20000,     # Top 20,000 features
    lowercase=True,
    min_df=5,               # Ignore patterns appearing in <5 URLs
    max_df=0.95             # Ignore patterns appearing in >95% of URLs
)



[7/8] Converting URLs to features (TF-IDF)...
  This may take 2-5 minutes for large datasets...


In [74]:
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"✓ Feature matrix: {X_train_tfidf.shape[0]:,} rows × {X_train_tfidf.shape[1]:,} features")
print(f"✓ Memory usage: {X_train_tfidf.data.nbytes / 1024**2:.1f} MB")

# ============================================
# STEP 8: TRAIN RANDOM FOREST
# ============================================
print("\n[8/8] Training Random Forest model...")
print("  This may take 10-30 minutes depending on your computer...")

✓ Feature matrix: 506,006 rows × 20,000 features
✓ Memory usage: 479.4 MB

[8/8] Training Random Forest model...
  This may take 10-30 minutes depending on your computer...


In [75]:
if len(X_train) > 500000:
    n_estimators = 100
elif len(X_train) > 200000:
    n_estimators = 150
else:
    n_estimators = 200

model = RandomForestClassifier(
    n_estimators=n_estimators,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',  # Handles any remaining imbalance
    n_jobs=-1,
    random_state=42,
    verbose=1
)

In [76]:
model.fit(X_train_tfidf, y_train)
print("✓ Model training complete!")

# ============================================
# EVALUATION
# ============================================
print("\n" + "="*60)
print("📊 MODEL EVALUATION")
print("="*60)

# Predictions
y_pred = model.predict(X_test_tfidf)
y_proba = model.predict_proba(X_test_tfidf)[:, 1]

# Accuracy
accuracy = accuracy_score(y_test, y_pred) * 100
print(f"\n🎯 ACCURACY: {accuracy:.2f}%")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:   28.5s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  1.5min finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.


✓ Model training complete!

📊 MODEL EVALUATION


[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.2s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.8s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.2s



🎯 ACCURACY: 99.71%


[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.9s finished


#### if accuracy >= 95:
    print(" GOAL ACHIEVED: 95%+ accuracy!")
elif accuracy >= 90:
    print(" CLOSE: Within 90-95% range")
else:
    print(" NEEDS IMPROVEMENT: Below 90%")

# Detailed metrics
print("\n DETAILED CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred, 
                          target_names=['Benign/Misc (0)', 'Malicious/Spam (1)']))


In [77]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\n📊 CONFUSION MATRIX:")
print("                 PREDICTED")
print("                 Benign  Malicious")
print(f"ACTUAL Benign    {cm[0,0]:6d}  {cm[0,1]:6d}")
print(f"ACTUAL Malicious {cm[1,0]:6d}  {cm[1,1]:6d}")

# Calculate additional metrics
tn, fp, fn, tp = cm.ravel()
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"\n📐 ADDITIONAL METRICS:")
print(f"  Precision: {precision*100:.2f}%")
print(f"  Recall:    {recall*100:.2f}%")
print(f"  F1-Score:  {f1*100:.2f}%")


📊 CONFUSION MATRIX:
                 PREDICTED
                 Benign  Malicious
ACTUAL Benign     63104     147
ACTUAL Malicious    222   63029

📐 ADDITIONAL METRICS:
  Precision: 99.77%
  Recall:    99.65%
  F1-Score:  99.71%


In [78]:
def get_malicious_score(url):
    """Returns score 0-100 for a URL"""
    url_features = vectorizer.transform([url])
    probability = model.predict_proba(url_features)[0][1]
    return round(probability * 100, 2)

In [79]:
def get_malicious_score(url):
    """Returns score 0-100 for a URL"""
    url_features = vectorizer.transform([url])
    probability = model.predict_proba(url_features)[0][1]
    return round(probability * 100, 2)

def classify_url(url):
    """Returns score and classification"""
    score = get_malicious_score(url)
    
    if score >= 70:
        verdict = "🔴 MALICIOUS"
        risk = "High"
    elif score >= 40:
        verdict = "🟠 SUSPICIOUS"
        risk = "Medium"
    else:
        verdict = "🟢 SAFE"
        risk = "Low"
    
    return score, verdict, risk

In [80]:
print("\n" + "="*60)
print("🔍 TESTING ON SAMPLE URLS")
print("="*60)

test_urls = [
    "https://google.com",
    "https://facebook.com",
    "http://free-gift-card-winner.com/claim",
    "https://paypal.com/login",
    "http://secure-verify-account.xyz/update"
]



🔍 TESTING ON SAMPLE URLS


In [83]:
for url in test_urls:
    score, verdict, risk = classify_url(url)
    print(f"\nURL: {url[:60]}...")
    print(f"Score: {score}/100 - {verdict} (Risk: {risk})")

# ============================================
# SAVE MODEL
# ============================================
print("\n" + "="*60)
print(" SAVING MODEL FOR PRODUCTION")
print("="*60)

import os
os.makedirs('models', exist_ok=True)

joblib.dump(model, 'models/url_classifier_model.pkl')
joblib.dump(vectorizer, 'models/url_vectorizer.pkl')

print("✓ Model saved to 'models/url_classifier_model.pkl'")
print("✓ Vectorizer saved to 'models/url_vectorizer.pkl'")

[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0


URL: https://google.com...
Score: 80.05/100 - 🔴 MALICIOUS (Risk: High)

URL: https://facebook.com...
Score: 82.43/100 - 🔴 MALICIOUS (Risk: High)

URL: http://free-gift-card-winner.com/claim...
Score: 99.97/100 - 🔴 MALICIOUS (Risk: High)

URL: https://paypal.com/login...
Score: 95.29/100 - 🔴 MALICIOUS (Risk: High)

URL: http://secure-verify-account.xyz/update...
Score: 99.83/100 - 🔴 MALICIOUS (Risk: High)

 SAVING MODEL FOR PRODUCTION
✓ Model saved to 'models/url_classifier_model.pkl'
✓ Vectorizer saved to 'models/url_vectorizer.pkl'


In [82]:
print("\n" + "="*60)
print(" TRAINING COMPLETE - SUMMARY")
print("="*60)
print(f"""
📊 DATASET STATISTICS:
   • Total URLs:        {len(df):,}
   • Class 0 (Benign):  {count_0:,} ({percentage_0:.1f}%)
   • Class 1 (Malicious): {count_1:,} ({percentage_1:.1f}%)
   • Balance Ratio:     {balance_ratio:.3f}

🎯 MODEL PERFORMANCE:
   • Accuracy:          {accuracy:.2f}%
   • Precision:         {precision*100:.2f}%
   • Recall:            {recall*100:.2f}%
   • F1-Score:          {f1*100:.2f}%

⚙️ MODEL CONFIGURATION:
   • Features:          {X_train_tfidf.shape[1]:,}
   • Trees:             {n_estimators}
   • Training size:     {len(X_train):,} URLs
   PRODUCTION READY:
   • Model file:        models/url_classifier_model.pkl
   • Vectorizer file:   models/url_vectorizer.pkl
   • API ready:         Yes
   • Extension ready:   Yes
""")

print("="*60)
print(" You can now deploy to production!")
print("="*60)


 TRAINING COMPLETE - SUMMARY

📊 DATASET STATISTICS:
   • Total URLs:        632,508
   • Class 0 (Benign):  316,254 (50.0%)
   • Class 1 (Malicious): 316,254 (50.0%)
   • Balance Ratio:     1.000

🎯 MODEL PERFORMANCE:
   • Accuracy:          99.71%
   • Precision:         99.77%
   • Recall:            99.65%
   • F1-Score:          99.71%

⚙️ MODEL CONFIGURATION:
   • Features:          20,000
   • Trees:             100
   • Training size:     506,006 URLs
   PRODUCTION READY:
   • Model file:        models/url_classifier_model.pkl
   • Vectorizer file:   models/url_vectorizer.pkl
   • API ready:         Yes
   • Extension ready:   Yes

 You can now deploy to production!


In [84]:
# Test normalization
test_urls = [
    "https://www.google.com",
    "http://google.com",
    "https://google.com/",
    "http://www.google.com/index.html"
]

for url in test_urls:
    normalized = normalize_url(url)
    print(f"Original: {url}")
    print(f"Normalized: {normalized}")
    print(f"Same? {normalize_url('https://google.com') == normalized}\n")

Original: https://www.google.com
Normalized: https://google.com
Same? True

Original: http://google.com
Normalized: http://google.com
Same? False

Original: https://google.com/
Normalized: https://google.com
Same? True

Original: http://www.google.com/index.html
Normalized: http://google.com/index.html
Same? False

